# Rental Prices and Income Inequality in Buenos Aires City

### Exploring the spatial relationship between household income and rental prices across CABA communes

**Facundo Gastón Sasso** · Sociologist, University of Buenos Aires (UBA)

---

## Executive summary

This project explores whether the spatial distribution of rental prices in Buenos Aires City follows the distribution of income across its 15 communes.

The analysis combines:

- **EAH 2023** individual-level data from Buenos Aires City, using family per-capita income;
- historical **rental-price data for 2018–2019**, aggregated by commune;
- official geographic boundaries for the city's communes.

The original exploratory analysis found a **positive Pearson correlation of 0.87** between average family per-capita income and average rental prices across the communes for which rental data were available. The analysis also identified important deviations from the overall spatial pattern, particularly in communes with heterogeneous urban and socioeconomic profiles.

> **Important limitation:** income and rental data do not refer to the same period. Income is from 2023, while the rental-price series used here covers 2018–2019. Therefore, this project should be interpreted as an **exploratory analysis of spatial patterns**, not as a contemporaneous measure of housing affordability or rent burden.

## Research question

> **Do rental prices follow the spatial distribution of income across Buenos Aires City's communes?**

### Initial hypothesis

The original hypothesis was that communes with higher resident incomes would also tend to show higher average rental prices.

The purpose of the analysis is not only to test whether the two variables move together, but also to identify **communes that deviate from the general spatial pattern**.

## Data

### 1. EAH 2023 — Household Survey

The analysis uses individual-level EAH 2023 data and the following variables:

- commune;
- family per-capita income;
- sex of respondent.

Only observations with positive family per-capita income are retained.

### 2. Rental prices

The rental dataset contains historical average rental prices by commune and year. The original project selected **2018 and 2019** and calculated the average rental price across those two years for each commune.

### 3. Geographic boundaries

Official Buenos Aires City commune boundaries are used to create choropleth maps.

The repository does **not** include the raw microdata. Place the two CSV files in `data/raw/` before running the notebook.

## Reproducibility and project structure

Expected files:

```text
data/
└── raw/
    ├── EAH_2023_ind.csv
    └── Precio alquiler_Ba_ciudad_csv.csv
```

The notebook is deliberately separated from the raw data so that potentially large public datasets do not need to be committed to GitHub.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr

DATA_DIR = Path("../data/raw")

EAH_FILE = DATA_DIR / "EAH_2023_ind.csv"
RENT_FILE = DATA_DIR / "Precio alquiler_Ba_ciudad_csv.csv"

sns.set_theme(style="whitegrid")

print("Working directory:", Path.cwd())
print("EAH file:", EAH_FILE)
print("Rental file:", RENT_FILE)

In [ ]:
# Fail with a useful message if the public data files have not been added yet.
missing = [str(p) for p in [EAH_FILE, RENT_FILE] if not p.exists()]

if missing:
    raise FileNotFoundError(
        "The following input files are missing:\n- " +
        "\n- ".join(missing) +
        "\n\nSee data/README.md for instructions."
    )

## 1. Income distribution in Buenos Aires City

In [ ]:
income_columns = [
    "Número de vivienda",
    "Número de hogar",
    "Número de miembro",
    "Comuna",
    "Sexo del entrevistado",
    "Monto del ingreso per cápita familiar (mes pasado; incluye valores imputados. Se considera sólo miembros que no son servicio doméstico)"
]

eah = pd.read_csv(EAH_FILE, usecols=income_columns)

eah.columns = [
    "num_vivienda",
    "num_hogar",
    "num_individuo",
    "comuna",
    "sexo",
    "ingreso_per_capita_familiar"
]

# Keep only positive income observations, following the original analysis.
eah = eah.loc[eah["ingreso_per_capita_familiar"] > 0].copy()

eah["comuna"] = pd.to_numeric(eah["comuna"], errors="coerce").astype("Int64")

print(f"Observations after filtering positive income: {len(eah):,}")
eah.head()

In [ ]:
income_mean = eah["ingreso_per_capita_familiar"].mean()
income_median = eah["ingreso_per_capita_familiar"].median()
income_std = eah["ingreso_per_capita_familiar"].std()

print(f"Mean:   ${income_mean:,.0f}")
print(f"Median: ${income_median:,.0f}")
print(f"Std.:   ${income_std:,.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

income_plot = eah.loc[
    eah["ingreso_per_capita_familiar"] < 2_000_000,
    "ingreso_per_capita_familiar"
]

ax.hist(income_plot, bins=50)
ax.axvline(income_mean, linestyle="--", label="Mean")
ax.axvline(income_median, linestyle="--", label="Median")

ax.set_title("Family per-capita income distribution — CABA, 2023")
ax.set_xlabel("Family per-capita income")
ax.set_ylabel("Number of observations")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
income_by_commune = (
    eah.groupby("comuna")["ingreso_per_capita_familiar"]
    .agg(["mean", "median", "std", "count"])
    .reset_index()
    .sort_values("mean")
)

income_by_commune

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

sns.barplot(
    data=income_by_commune,
    x="mean",
    y="comuna",
    errorbar=None,
    ax=ax
)

ax.set_title("Average family per-capita income by commune — CABA, 2023")
ax.set_xlabel("Average family per-capita income")
ax.set_ylabel("Commune")

plt.tight_layout()
plt.show()

### What this stage establishes

The income distribution provides the socioeconomic spatial context for the second part of the analysis.

In the original run, **Commune 14 had the highest average income**, while **Commune 8 had the lowest average income**. The original data also showed substantial dispersion within several communes, which is important when interpreting commune-level averages.

## 2. Rental prices by commune

In [ ]:
rent = pd.read_csv(RENT_FILE, sep=";")

print("Shape:", rent.shape)
print("Columns:", rent.columns.tolist())
rent.head()

In [ ]:
rent = rent.dropna(subset=["precio_prom"]).copy()

rent["comuna"] = pd.to_numeric(rent["comuna"], errors="coerce")
rent["anio"] = pd.to_numeric(rent["anio"], errors="coerce")

rent_2018_2019 = rent.loc[
    rent["anio"].isin([2018, 2019]),
    ["comuna", "precio_prom", "anio"]
].copy()

rental_by_commune = (
    rent_2018_2019
    .groupby(["comuna", "anio"])["precio_prom"]
    .mean()
    .unstack()
)

rental_by_commune["precio_prom_alquiler_2018_2019"] = rental_by_commune.mean(axis=1)

rental_by_commune = (
    rental_by_commune[["precio_prom_alquiler_2018_2019"]]
    .reset_index()
    .sort_values("precio_prom_alquiler_2018_2019")
)

rental_by_commune

### Missing rental data

The historical rental dataset does not contain observations for **Commune 8** in the 2018–2019 period.

Rather than silently deleting the commune, the final version of the analysis keeps Commune 8 in the income table and represents its rental value as missing. This makes the limitation explicit and prevents the geographic merge from producing misleading matches.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

sns.barplot(
    data=rental_by_commune,
    x="precio_prom_alquiler_2018_2019",
    y="comuna",
    errorbar=None,
    ax=ax
)

ax.set_title("Average rental price by commune — 2018–2019")
ax.set_xlabel("Average rental price")
ax.set_ylabel("Commune")

plt.tight_layout()
plt.show()

## 3. Combining income and rental prices

In [ ]:
analysis = income_by_commune.merge(
    rental_by_commune,
    on="comuna",
    how="left",
    validate="one_to_one",
    indicator=True
)

analysis["rental_data_available"] = analysis["_merge"].eq("both")
analysis = analysis.drop(columns="_merge")

analysis.sort_values("comuna")

In [ ]:
available = analysis.loc[analysis["rental_data_available"]].copy()

correlation = available["mean"].corr(
    available["precio_prom_alquiler_2018_2019"],
    method="pearson"
)

r, p_value = pearsonr(
    available["mean"],
    available["precio_prom_alquiler_2018_2019"]
)

print(f"Communes with rental data: {len(available)}")
print(f"Pearson correlation: {correlation:.2f}")
print(f"p-value: {p_value:.4g}")

### Interpreting the correlation

In the original analysis, the Pearson correlation was **0.87** across the 14 communes with rental data.

This indicates a strong positive linear association at the commune level in the observed data: communes with higher average family per-capita income also tended to have higher average rental prices.

However, correlation should not be interpreted as causation. More importantly, the two datasets refer to different periods, so the result describes a **spatial association across two non-contemporaneous datasets**.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

sns.scatterplot(
    data=available,
    x="mean",
    y="precio_prom_alquiler_2018_2019",
    size="std",
    sizes=(70, 700),
    legend=False,
    ax=ax
)

for _, row in available.iterrows():
    ax.annotate(
        f"{int(row['comuna'])}",
        (row["mean"], row["precio_prom_alquiler_2018_2019"]),
        xytext=(6, 4),
        textcoords="offset points",
        fontsize=9
    )

ax.set_title("Income and rental prices across CABA communes")
ax.set_xlabel("Average family per-capita income — EAH 2023")
ax.set_ylabel("Average rental price — 2018–2019")

plt.tight_layout()
plt.show()

## 4. Identifying spatial deviations

The scatterplot shows the general positive association, but the most interesting cases are the communes that sit relatively far from the overall pattern.

These deviations should be treated as **descriptive signals for further research**, not as explanations in themselves.

In [ ]:
# A simple descriptive residual from a linear fit.
# This is used only to identify communes that deviate from the overall
# observed relationship; it is not presented as a causal model.

slope, intercept = np.polyfit(
    available["mean"],
    available["precio_prom_alquiler_2018_2019"],
    1
)

available["predicted_rent"] = (
    intercept + slope * available["mean"]
)

available["rent_deviation"] = (
    available["precio_prom_alquiler_2018_2019"]
    - available["predicted_rent"]
)

deviations = (
    available[
        ["comuna", "mean", "precio_prom_alquiler_2018_2019", "rent_deviation"]
    ]
    .sort_values("rent_deviation")
)

deviations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

plot_dev = deviations.sort_values("rent_deviation")

sns.barplot(
    data=plot_dev,
    x="rent_deviation",
    y="comuna",
    errorbar=None,
    ax=ax
)

ax.axvline(0, linestyle="--")
ax.set_title("Deviation of rental prices from the income-based spatial pattern")
ax.set_xlabel("Observed rental price minus predicted rental price")
ax.set_ylabel("Commune")

plt.tight_layout()
plt.show()

## 5. Spatial visualization

The geographic analysis uses official Buenos Aires City commune boundaries.

A key methodological correction from the original notebook is to merge the geographic data using the **`comuna` column explicitly**, rather than joining the commune geometry to the analytical dataframe index. This prevents commune labels from becoming misaligned in the maps.

In [ ]:
import geopandas as gpd

GEO_URL = (
    "https://data.buenosaires.gob.ar/dataset/"
    "comunas/resource/b0b627ac-5b47-4574-89ac-6999b63598ee"
)

# Direct GeoJSON endpoint used by the Buenos Aires City open-data resource.
GEOJSON_URL = (
    "https://cdn.buenosaires.gob.ar/datosabiertos/"
    "datasets/ministerio-de-educacion/comunas/comunas.geojson"
)

communes_geo = gpd.read_file(GEOJSON_URL)

communes_geo["comuna"] = pd.to_numeric(
    communes_geo["comuna"], errors="coerce"
)

geo_analysis = communes_geo.merge(
    analysis,
    on="comuna",
    how="left",
    validate="one_to_one"
)

print("Communes in geographic layer:", sorted(communes_geo["comuna"].dropna().unique()))
print("Communes in analytical layer:", sorted(analysis["comuna"].dropna().unique()))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

geo_analysis.plot(
    column="precio_prom_alquiler_2018_2019",
    cmap="magma",
    legend=True,
    missing_kwds={
        "color": "lightgrey",
        "edgecolor": "white",
        "label": "No rental data"
    },
    edgecolor="white",
    linewidth=0.8,
    ax=ax
)

ax.set_title("Average rental price by commune — 2018–2019")
ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

geo_analysis.plot(
    column="mean",
    cmap="viridis",
    legend=True,
    edgecolor="white",
    linewidth=0.8,
    ax=ax
)

ax.set_title("Average family per-capita income by commune — 2023")
ax.set_axis_off()

plt.tight_layout()
plt.show()

## 6. Main findings

Based on the original notebook and its observed results:

1. **Income and rental prices show a strong positive spatial association.**  
   The original run produced a Pearson correlation of **0.87** across the 14 communes with rental data.

2. **The highest-income commune was Commune 14**, while Commune 8 had the lowest average income in the EAH 2023 data used here.

3. **Commune 14 also had the highest average rental price** in the 2018–2019 rental dataset.

4. **Commune 8 is a relevant limitation rather than a result.**  
   It has income observations but no rental-price observations in the selected historical period, so it cannot be included in the correlation or rental-price ranking.

5. **Commune-level averages hide internal heterogeneity.**  
   The original analysis found substantial standard deviations in several communes. Commune 1 and Commune 14 were particularly notable because of their high dispersion in income.

6. **The relationship deserves a more detailed follow-up.**  
   Commune 1, for example, combines very different urban areas and should not be interpreted as a homogeneous socioeconomic space simply from its commune-level average.

## Limitations

### Temporal mismatch

The largest limitation is that the income and rental datasets do not refer to the same period:

- Income: **2023**
- Rental prices: **2018–2019**

Therefore, the analysis does not estimate current affordability or the share of income devoted to rent.

### Spatial aggregation

The unit of analysis is the **commune**, which can conceal considerable socioeconomic and housing-market heterogeneity within each commune.

### Rental-data coverage

Commune 8 has no rental observation in the selected 2018–2019 data. This means the correlation is calculated over 14 communes.

### Descriptive analysis

The project identifies spatial associations and deviations. It does not establish causal relationships between income, rental prices, migration, neighborhood characteristics, accessibility, or other possible mechanisms.

## Next steps

A stronger version of the analysis could:

- obtain rental data for 2023 or a period closer to the EAH;
- compare rental prices by **neighborhood** rather than only commune;
- distinguish housing typologies and number of rooms;
- calculate rent-to-income indicators using temporally comparable datasets;
- examine spatial accessibility and transport;
- test whether socioeconomic heterogeneity within communes explains deviations from the overall pattern.

These extensions would turn the project from an exploratory spatial comparison into a more complete housing-market analysis.

## Tools

**Python · Pandas · NumPy · Matplotlib · Seaborn · GeoPandas · SciPy**

**Research methods:** exploratory data analysis · data cleaning · aggregation · spatial analysis · descriptive statistics · correlation analysis · data visualization